# 🌸 PureGlow AI — Recommendation Model
**Notebook:** `03_recommendation_model.ipynb`

Builds the core recommendation engine:
- **Skincare** → matches products to skin type + concerns
- **Makeup** → matches colours to skin tone + undertone

**Outputs:**
- `models/skincare_products.csv` — scored product database
- `models/makeup_products.csv` — colour-tagged product database  
- `models/colour_config.json` — undertone → colour palette mapping
- `src/recommendation_engine.py` — importable Python module for the API

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

# ── Paths ─────────────────────────────────────────────────────────────────────
CLEAN_DIR  = Path('../data/cleaned').resolve()
MODELS_DIR = Path('../models').resolve()
SRC_DIR    = Path('../src').resolve()
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

print(f'Cleaned data : {CLEAN_DIR}')
print(f'Models output: {MODELS_DIR}')
print(f'Src output   : {SRC_DIR}')

def load(filename):
    path = CLEAN_DIR / filename
    if not path.exists():
        print(f'  ⚠  Not found: {filename}')
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'  ✓ {filename:<45} {df.shape[0]:>8,} rows × {df.shape[1]:>3} cols')
    return df

print('\n✅ Setup complete!')

Cleaned data : C:\Users\HP\OneDrive\Desktop\PureGlow AI\data\cleaned
Models output: C:\Users\HP\OneDrive\Desktop\PureGlow AI\models
Src output   : C:\Users\HP\OneDrive\Desktop\PureGlow AI\src

✅ Setup complete!


In [2]:
# ── Load Cleaned Data ─────────────────────────────────────────────────────────
print('Loading datasets...\n')

skincare     = load('02_skincare_products_clean.csv')
sephora      = load('07_sephora_clean.csv')
makeup       = load('09_makeup_clean.csv')
ingredients  = load('10_ingredients_clean.csv')
product_info = load('05_product_info_clean.csv')
rankings     = load('04_popularity_rankings_clean.csv')
reviews      = load('01_reviews_clean.csv')

print('\n✅ All datasets loaded!')

Loading datasets...

  ✓ 02_skincare_products_clean.csv                   3,474 rows ×  49 cols
  ✓ 07_sephora_clean.csv                             2,179 rows ×  20 cols
  ✓ 09_makeup_clean.csv                                931 rows ×  19 cols
  ✓ 10_ingredients_clean.csv                        66,417 rows ×   2 cols
  ✓ 05_product_info_clean.csv                        8,494 rows ×  27 cols
  ✓ 04_popularity_rankings_clean.csv                   400 rows ×   8 cols
  ✓ 01_reviews_clean.csv                          1,094,411 rows ×  19 cols

✅ All datasets loaded!


---
## Step 1: Inspect Columns
Before building the model, we need to understand what columns we have.
The recommendation engine will use whichever columns match your real data.

In [3]:
# ── Column Inspector ─────────────────────────────────────────────────────────
# This tells you EXACTLY what columns you have in each dataset
# Use this to adjust the COLUMN MAPPING in Step 2

for name, df in [('SKINCARE', skincare), ('SEPHORA', sephora),
                  ('MAKEUP', makeup), ('PRODUCT_INFO', product_info)]:
    if df.empty:
        continue
    print(f'\n{"─"*60}')
    print(f'  {name}  ({len(df):,} rows × {len(df.columns)} cols)')
    print(f'  Columns: {list(df.columns)}')
    print(f'  Sample:')
    display(df.head(2))


────────────────────────────────────────────────────────────
  SKINCARE  (3,474 rows × 49 cols)
  Columns: ['unnamed_0', 'brand', 'name', 'price', 'n_of_reviews', 'n_of_loves', 'review_score', 'size', 'clean_product', 'category_anti_aging', 'category_bb__cc_cream', 'category_bath__shower', 'category_beauty_supplements', 'category_blemish__acne_treatments', 'category_blotting_papers', 'category_body_lotions__body_oils', 'category_cellulite__stretch_marks', 'category_decollete__neck_creams', 'category_exfoliators', 'category_eye_creams__treatments', 'category_eye_masks', 'category_face_masks', 'category_face_oils', 'category_face_primer', 'category_face_serums', 'category_face_sunscreen', 'category_face_wash__cleansers', 'category_facial_peels', 'category_foundation', 'category_hair_oil', 'category_highlighter', 'category_holistic_wellness', 'category_mini_size', 'category_mists__essences', 'category_moisturizer__treatments', 'category_moisturizers', 'category_night_creams', 'category_s

,unnamed_0,brand,name,price,n_of_reviews,n_of_loves,review_score,size,clean_product,category_anti_aging,category_bb__cc_cream,category_bath__shower,category_beauty_supplements,category_blemish__acne_treatments,category_blotting_papers,category_body_lotions__body_oils,category_cellulite__stretch_marks,category_decollete__neck_creams,category_exfoliators,category_eye_creams__treatments,category_eye_masks,category_face_masks,category_face_oils,category_face_primer,category_face_serums,category_face_sunscreen,category_face_wash__cleansers,category_facial_peels,category_foundation,category_hair_oil,category_highlighter,category_holistic_wellness,category_mini_size,category_mists__essences,category_moisturizer__treatments,category_moisturizers,category_night_creams,category_setting_spray__powder,category_sheet_masks,category_skincare,category_tinted_moisturizer,category_toners,category_tools,category_value__gift_sets,reviews_to_loves_ratio,return_on_reviews,price_per_ounce,sub_category,category
0,0,Drunk Elephant,Protini Polypeptide Moisturizer,NaN,1000,136008,4.2097,NaN,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.74,0.42,40.24,skincare,NaN
1,1,La Mer,Crreme de la Mer,NaN,493,61648,4.0974,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.80,0.83,175.00,skincare,NaN



────────────────────────────────────────────────────────────
  SEPHORA  (2,179 rows × 20 cols)
  Columns: ['cosmetic_link', 'brand_name', 'cosmetic_name', 'num_customer', 'price', 'ingredients', 'about', 'reviews', 'recommended', 'what_it_is', 'skin_type', 'skincare_concerns', 'formulation', 'benefits', 'highlighted_ingredients', 'ingredient_callouts', 'what_else_you_need_to_know', 'clinical_results', 'clean_ingredients', 'new_ingredients']
  Sample:


,cosmetic_link,brand_name,cosmetic_name,num_customer,price,ingredients,about,reviews,recommended,what_it_is,skin_type,skincare_concerns,formulation,benefits,highlighted_ingredients,ingredient_callouts,what_else_you_need_to_know,clinical_results,clean_ingredients,new_ingredients
0,https://www.sephora.com/product/summer-fridays-lip-butter-balm-P455936?skuId...,Summer Fridays,Lip Butter Balm for Hydration & Shine,6.7K,24.0,"-Shea and Murumuru Seed Butters: Natural moisturizers that soothe, relieve, ...",What it is: A silky vegan balm that hydrates and soothes dry lips in seconds...,4.4,86%,A silky vegan balm that hydrates and soothes dry lips in seconds.,NaN,Dryness and Dullness,NaN,NaN,NaN,"This product is vegan, gluten-free, cruelty-free, and comes in recyclable pa...",This formula delivers soothing moisture to parched lips in seconds. Butter u...,"In an independent clinical study, with 39 participants aged 20 - 55:",NaN,"Phytosteryl/Behenyl Dimer Dilinoleate, Diisostearyl Malate, Hydrogenated Pol..."
1,https://www.sephora.com/product/glow-recipe-watermelon-glow-pha-bha-pore-tig...,Glow Recipe,Watermelon Glow PHA + BHA Pore-Tight Toner,6.1K,NaN,"-Watermelon Extract: Hydrates, soothes, and delivers essential vitamins and ...","What it is: A bestselling, gentle PHA- and BHA-infused watermelon toner that...",4.3,84%,"A bestselling, gentle PHA- and BHA-infused watermelon toner that hydrates, g...","Normal, Dry, Combination, and Oily","Pores, Dryness, and Dullness",Lightweight Liquid,NaN,NaN,"This product is vegan, cruelty-free, and comes in recyclable packaging.","Suitable for all skin types, this bouncy, alcohol-free toner is made with PH...",Based on instrumental testing on 31 women when used as directed After 2 week...,NaN,"Opuntia Ficus-Indica Stem Extract, Citrullus Lanatus (Watermelon) Fruit Extr..."



────────────────────────────────────────────────────────────
  MAKEUP  (931 rows × 19 cols)
  Columns: ['id', 'brand', 'name', 'price', 'price_sign', 'currency', 'image_link', 'product_link', 'website_link', 'description', 'rating', 'category', 'product_type', 'tag_list', 'created_at', 'updated_at', 'product_api_url', 'api_featured_image', 'product_colors']
  Sample:


,id,brand,name,price,price_sign,currency,image_link,product_link,website_link,description,rating,category,product_type,tag_list,created_at,updated_at,product_api_url,api_featured_image,product_colors
0,1048,colourpop,Lippie Pencil,5.0,NaN,CAD,https://cdn.shopify.com/s/files/1/1338/0845/collections/lippie-pencil_grande...,https://colourpop.com/collections/lippie-pencil,https://colourpop.com,Lippie Pencil A long-wearing and high-intensity lip pencil that glides on ea...,NaN,pencil,lip_liner,NaN,2018-07-08T23:45:08.056Z,2018-07-09T00:53:23.301Z,http://makeup-api.herokuapp.com/api/v1/products/1048.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/048/or...,NaN
1,1047,colourpop,Blotted Lip,5.5,NaN,CAD,https://cdn.shopify.com/s/files/1/1338/0845/products/brain-freeze_a_800x1200...,https://colourpop.com/collections/lippie-stix?filter=blotted-lip,https://colourpop.com,Blotted Lip Sheer matte lipstick that creates the perfect popsicle pout! For...,NaN,lipstick,lipstick,NaN,2018-07-08T22:01:20.178Z,2018-07-09T00:53:23.287Z,http://makeup-api.herokuapp.com/api/v1/products/1047.json,//s3.amazonaws.com/donovanbailey/products/api_featured_images/000/001/047/or...,NaN



────────────────────────────────────────────────────────────
  PRODUCT_INFO  (8,494 rows × 27 cols)
  Columns: ['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']
  Sample:


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,variation_desc,ingredients,price_usd,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,NaN,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance...",35.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scent', 'Woody & Earthy Scent', 'F...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Ethylhexyl Methoxycinn...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Warm &Spicy Scent']",Fragrance,Women,Perfume,2,85.0,30.0


---
## Step 2: Column Mapping
⚠️ **Adjust these mappings to match your actual column names** (from Step 1 above).

In [4]:
# ── COLUMN MAPPING ────────────────────────────────────────────────────────────
# ⚠️  Edit these to match your real column names from Step 1

SKINCARE_COLS = {
    'name':        'name',               # product name
    'brand':       'brand',              # brand
    'category':    'source_category',    # moisturizer / treatment / eyecare etc.
    'price':       'price',              # numeric price
    'rating':      'rating',             # numeric rating 0-5
    'skin_type':   'skin_type',          # oily/dry/combination/sensitive
    'ingredients': 'ingredients',        # text list of ingredients
    'concerns':    'highlights',         # acne/dark spots/redness/dullness
    'url':         'url',                # product page URL
}

MAKEUP_COLS = {
    'name':         'name',              # product name
    'brand':        'brand',             # brand
    'product_type': 'product_type',      # lipstick/blush/eyeshadow/highlighter
    'price':        'price',             # numeric price
    'rating':       'rating',            # numeric rating 0-5
    'hex_color':    'hex_value',         # hex colour code e.g. #FF5733
    'color_name':   'name',              # colour name e.g. Rose Gold
    'tag':          'tag',               # additional tags
    'url':          'product_link',      # product URL
}

SEPHORA_COLS = {
    'name':         'name',
    'brand':        'brand_name',
    'category':     'primary_category',
    'sub_category': 'secondary_category',
    'price':        'price_usd',
    'rating':       'rating',
    'skin_type':    'highlights',        # or 'skin_type' or 'tags'
    'url':          'url',
}

print('✅ Column mappings defined')
print('\n⚠️  If you see KeyErrors below, adjust the mappings above to match your column names')

✅ Column mappings defined

⚠️  If you see KeyErrors below, adjust the mappings above to match your column names


---
## Step 3: Build Master Product Table

In [5]:
# ── Helper: safely get a column ───────────────────────────────────────────────
def get_col(df, mapping_key, mapping_dict, default=np.nan):
    col = mapping_dict.get(mapping_key, '')
    if col and col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)

# ── Build Skincare Master ─────────────────────────────────────────────────────
skincare_frames = []

if not skincare.empty:
    s = pd.DataFrame()
    s['product_id']    = 'sk_' + skincare.index.astype(str)
    s['name']          = get_col(skincare, 'name',       SKINCARE_COLS, 'Unknown')
    s['brand']         = get_col(skincare, 'brand',      SKINCARE_COLS, 'Unknown')
    s['category']      = get_col(skincare, 'category',   SKINCARE_COLS, 'skincare')
    s['price']         = pd.to_numeric(get_col(skincare, 'price',  SKINCARE_COLS), errors='coerce')
    s['rating']        = pd.to_numeric(get_col(skincare, 'rating', SKINCARE_COLS), errors='coerce')
    s['skin_type_raw'] = get_col(skincare, 'skin_type',   SKINCARE_COLS, '')
    s['concerns_raw']  = get_col(skincare, 'concerns',    SKINCARE_COLS, '')
    s['ingredients']   = get_col(skincare, 'ingredients', SKINCARE_COLS, '')
    s['url']           = get_col(skincare, 'url',         SKINCARE_COLS, '')
    s['source']        = 'skincare_db'
    s['product_kind']  = 'skincare'
    skincare_frames.append(s)
    print(f'✓ Skincare source added: {len(s):,} rows')

if not sephora.empty:
    s = pd.DataFrame()
    s['product_id']    = 'sep_' + sephora.index.astype(str)
    s['name']          = get_col(sephora, 'name',     SEPHORA_COLS, 'Unknown')
    s['brand']         = get_col(sephora, 'brand',    SEPHORA_COLS, 'Unknown')
    s['category']      = get_col(sephora, 'category', SEPHORA_COLS, 'general')
    s['price']         = pd.to_numeric(get_col(sephora, 'price',  SEPHORA_COLS), errors='coerce')
    s['rating']        = pd.to_numeric(get_col(sephora, 'rating', SEPHORA_COLS), errors='coerce')
    s['skin_type_raw'] = get_col(sephora, 'skin_type', SEPHORA_COLS, '')
    s['concerns_raw']  = pd.Series([''] * len(sephora), index=sephora.index)
    s['ingredients']   = pd.Series([''] * len(sephora), index=sephora.index)
    s['url']           = get_col(sephora, 'url', SEPHORA_COLS, '')
    s['source']        = 'sephora'
    s['product_kind']  = 'skincare'
    skincare_frames.append(s)
    print(f'✓ Sephora source added: {len(s):,} rows')

if skincare_frames:
    master_skincare = pd.concat(skincare_frames, ignore_index=True)
    master_skincare = master_skincare.dropna(subset=['name'])
    master_skincare = master_skincare[master_skincare['name'] != 'Unknown']
    print(f'\n✅ Master skincare table: {len(master_skincare):,} products')
    display(master_skincare.head(3))
else:
    master_skincare = pd.DataFrame()
    print('⚠️  No skincare data available')

✓ Skincare source added: 3,474 rows
✓ Sephora source added: 2,179 rows

✅ Master skincare table: 3,474 products


,product_id,name,brand,category,price,rating,skin_type_raw,concerns_raw,ingredients,url,source,product_kind
0,sk_0,Protini Polypeptide Moisturizer,Drunk Elephant,skincare,NaN,NaN,,,,,skincare_db,skincare
1,sk_1,Crreme de la Mer,La Mer,skincare,NaN,NaN,,,,,skincare_db,skincare
2,sk_2,CC+ Cream with SPF 50+,IT Cosmetics,skincare,NaN,NaN,,,,,skincare_db,skincare


In [6]:
# ── Build Makeup Master ───────────────────────────────────────────────────────
if not makeup.empty:
    m = pd.DataFrame()
    m['product_id']   = 'mk_' + makeup.index.astype(str)
    m['name']         = get_col(makeup, 'name',         MAKEUP_COLS, 'Unknown')
    m['brand']        = get_col(makeup, 'brand',        MAKEUP_COLS, 'Unknown')
    m['product_type'] = get_col(makeup, 'product_type', MAKEUP_COLS, 'makeup')
    m['price']        = pd.to_numeric(get_col(makeup, 'price',  MAKEUP_COLS), errors='coerce')
    m['rating']       = pd.to_numeric(get_col(makeup, 'rating', MAKEUP_COLS), errors='coerce')
    m['hex_color']    = get_col(makeup, 'hex_color',    MAKEUP_COLS, '')
    m['color_name']   = get_col(makeup, 'color_name',   MAKEUP_COLS, '')
    m['tag']          = get_col(makeup, 'tag',          MAKEUP_COLS, '')
    m['url']          = get_col(makeup, 'url',          MAKEUP_COLS, '')
    m['source']       = 'makeup_db'
    m['product_kind'] = 'makeup'
    master_makeup = m.dropna(subset=['name'])
    print(f'✅ Master makeup table: {len(master_makeup):,} products')
    display(master_makeup.head(3))
else:
    master_makeup = pd.DataFrame()
    print('⚠️  No makeup data available')

✅ Master makeup table: 931 products


,product_id,name,brand,product_type,price,rating,hex_color,color_name,tag,url,source,product_kind
0,mk_0,Lippie Pencil,colourpop,lip_liner,5.0,NaN,,Lippie Pencil,,https://colourpop.com/collections/lippie-pencil,makeup_db,makeup
1,mk_1,Blotted Lip,colourpop,lipstick,5.5,NaN,,Blotted Lip,,https://colourpop.com/collections/lippie-stix?filter=blotted-lip,makeup_db,makeup
2,mk_2,Lippie Stix,colourpop,lipstick,5.5,NaN,,Lippie Stix,,https://colourpop.com/collections/lippie-stix,makeup_db,makeup


---
## Step 4: Skin Type & Concern Tagging
Convert free-text fields into standardised flags the scoring engine can use.

In [7]:
# ── Standardised keyword maps ─────────────────────────────────────────────────
SKIN_TYPE_KEYWORDS = {
    'oily':        ['oily', 'oil control', 'oil-free', 'mattify', 'matte'],
    'dry':         ['dry', 'hydrating', 'hydration', 'moisture', 'moisturising', 'nourishing'],
    'combination': ['combination', 'balanced', 'normal'],
    'sensitive':   ['sensitive', 'gentle', 'soothing', 'calming', 'fragrance-free', 'hypoallergenic'],
}

CONCERN_KEYWORDS = {
    'acne':       ['acne', 'breakout', 'blemish', 'pore', 'salicylic', 'benzoyl', 'anti-acne', 'spot'],
    'dark_spots': ['dark spot', 'brightening', 'vitamin c', 'niacinamide', 'hyperpigmentation',
                   'uneven', 'discoloration', 'fade', 'kojic'],
    'redness':    ['redness', 'rosacea', 'anti-inflammatory', 'centella', 'green tea',
                   'calming', 'soothing', 'sensitive'],
    'dullness':   ['dull', 'glow', 'radiant', 'brightening', 'exfoliat', 'aha', 'bha',
                   'vitamin c', 'resurfac'],
    'aging':      ['anti-aging', 'anti-wrinkle', 'retinol', 'firming', 'lifting',
                   'collagen', 'peptide', 'fine line'],
    'dryness':    ['dry', 'moisture', 'hydrat', 'hyaluronic', 'ceramide', 'barrier'],
}

def tag_column(series, keyword_map):
    """Returns a dict of {tag: bool_series} for each keyword category."""
    text = series.fillna('').str.lower()
    return {
        tag: text.apply(lambda t: any(kw in t for kw in kws))
        for tag, kws in keyword_map.items()
    }

# Apply tagging to skincare master
if not master_skincare.empty:
    search_text = (master_skincare['name'].fillna('') + ' ' +
                   master_skincare['skin_type_raw'].fillna('') + ' ' +
                   master_skincare['concerns_raw'].fillna('') + ' ' +
                   master_skincare['ingredients'].fillna('')).str.lower()

    skin_tags  = tag_column(search_text, SKIN_TYPE_KEYWORDS)
    concern_tags = tag_column(search_text, CONCERN_KEYWORDS)

    for tag, series in skin_tags.items():
        master_skincare[f'skin_{tag}'] = series

    for tag, series in concern_tags.items():
        master_skincare[f'concern_{tag}'] = series

    # Coverage stats
    print('Skin type tag coverage:')
    for tag in skin_tags:
        pct = master_skincare[f'skin_{tag}'].mean() * 100
        print(f'  skin_{tag:<15} {pct:>5.1f}%  ({master_skincare[f"skin_{tag}"].sum():,} products)')

    print('\nConcern tag coverage:')
    for tag in concern_tags:
        pct = master_skincare[f'concern_{tag}'].mean() * 100
        print(f'  concern_{tag:<15} {pct:>5.1f}%  ({master_skincare[f"concern_{tag}"].sum():,} products)')
    
    print(f'\n✅ Tagged {len(master_skincare):,} skincare products')

Skin type tag coverage:
  skin_oily              3.3%  (114 products)
  skin_dry               8.2%  (285 products)
  skin_combination       1.2%  (40 products)
  skin_sensitive         1.4%  (49 products)

Concern tag coverage:
  concern_acne              6.9%  (241 products)
  concern_dark_spots        6.8%  (235 products)
  concern_redness           1.8%  (64 products)
  concern_dullness         10.5%  (366 products)
  concern_aging            11.2%  (390 products)
  concern_dryness           9.7%  (338 products)

✅ Tagged 3,474 skincare products


---
## Step 5: Makeup Colour → Undertone Mapping
Assign each makeup product to warm / cool / neutral undertone palettes.

In [8]:
# ── Undertone colour config ───────────────────────────────────────────────────
COLOUR_CONFIG = {
    'warm': {
        'description': 'Golden, peachy, yellow undertones',
        'blush':       ['peach', 'coral', 'terracotta', 'apricot', 'bronze',  'warm pink'],
        'lipstick':    ['coral', 'orange-red', 'brick', 'nude peach', 'copper', 'warm red', 'salmon'],
        'eyeshadow':   ['bronze', 'gold', 'copper', 'warm brown', 'rust', 'terracotta', 'peach'],
        'highlighter': ['gold', 'champagne', 'bronze', 'warm pearl'],
        'foundation':  ['yellow-based', 'warm beige', 'golden', 'warm ivory'],
    },
    'cool': {
        'description': 'Pink, red, blue undertones',
        'blush':       ['rose', 'pink', 'berry', 'mauve', 'fuchsia', 'cool pink'],
        'lipstick':    ['pink', 'berry', 'plum', 'fuchsia', 'cool red', 'burgundy', 'lavender'],
        'eyeshadow':   ['silver', 'taupe', 'cool brown', 'navy', 'grey', 'plum', 'lavender'],
        'highlighter': ['silver', 'icy pink', 'pearl', 'cool champagne'],
        'foundation':  ['pink-based', 'cool beige', 'rosy', 'cool ivory'],
    },
    'neutral': {
        'description': 'Balance of warm and cool, most versatile',
        'blush':       ['dusty rose', 'nude pink', 'soft peach', 'natural', 'sheer pink'],
        'lipstick':    ['nude', 'mauve', 'soft pink', 'natural rose', 'your-lips-but-better'],
        'eyeshadow':   ['warm taupe', 'champagne', 'soft brown', 'nude', 'satin', 'beige'],
        'highlighter': ['golden pearl', 'satin', 'warm silver', 'blush gold'],
        'foundation':  ['neutral beige', 'balanced', 'true beige', 'neutral ivory'],
    }
}

# Save colour config for API use
config_path = MODELS_DIR / 'colour_config.json'
with open(config_path, 'w') as f:
    json.dump(COLOUR_CONFIG, f, indent=2)
print(f'✅ Colour config saved → {config_path}')

# Show the palette
for undertone, data in COLOUR_CONFIG.items():
    print(f'\n  {undertone.upper()} ({data["description"]})')
    for product_type, shades in data.items():
        if product_type == 'description':
            continue
        print(f'    {product_type:<15}: {", ".join(shades[:4])}')

✅ Colour config saved → C:\Users\HP\OneDrive\Desktop\PureGlow AI\models\colour_config.json

  WARM (Golden, peachy, yellow undertones)
    blush          : peach, coral, terracotta, apricot
    lipstick       : coral, orange-red, brick, nude peach
    eyeshadow      : bronze, gold, copper, warm brown
    highlighter    : gold, champagne, bronze, warm pearl
    foundation     : yellow-based, warm beige, golden, warm ivory

  COOL (Pink, red, blue undertones)
    blush          : rose, pink, berry, mauve
    lipstick       : pink, berry, plum, fuchsia
    eyeshadow      : silver, taupe, cool brown, navy
    highlighter    : silver, icy pink, pearl, cool champagne
    foundation     : pink-based, cool beige, rosy, cool ivory

  NEUTRAL (Balance of warm and cool, most versatile)
    blush          : dusty rose, nude pink, soft peach, natural
    lipstick       : nude, mauve, soft pink, natural rose
    eyeshadow      : warm taupe, champagne, soft brown, nude
    highlighter    : golden pea

In [9]:
# ── Tag makeup products with undertone suitability ────────────────────────────
UNDERTONE_KEYWORDS = {
    'warm':    ['warm', 'golden', 'peach', 'coral', 'bronze', 'copper',
                'terracotta', 'apricot', 'gold', 'orange', 'amber', 'rust'],
    'cool':    ['cool', 'pink', 'berry', 'plum', 'silver', 'icy', 'rose',
                'lavender', 'mauve', 'fuchsia', 'burgundy', 'navy'],
    'neutral': ['nude', 'natural', 'neutral', 'balanced', 'classic', 'sheer',
                'taupe', 'beige', 'sand', 'soft', 'universal'],
}

if not master_makeup.empty:
    search_text = (master_makeup['name'].fillna('') + ' ' +
                   master_makeup['color_name'].fillna('') + ' ' +
                   master_makeup['tag'].fillna('')).str.lower()

    undertone_tags = tag_column(search_text, UNDERTONE_KEYWORDS)
    for tag, series in undertone_tags.items():
        master_makeup[f'undertone_{tag}'] = series

    # Products with no undertone match → assign neutral
    no_match = ~(master_makeup['undertone_warm'] |
                 master_makeup['undertone_cool'] |
                 master_makeup['undertone_neutral'])
    master_makeup.loc[no_match, 'undertone_neutral'] = True

    print('Undertone tag coverage:')
    for tag in UNDERTONE_KEYWORDS:
        pct = master_makeup[f'undertone_{tag}'].mean() * 100
        n   = master_makeup[f'undertone_{tag}'].sum()
        print(f'  undertone_{tag:<10} {pct:>5.1f}%  ({n:,} products)')

    print(f'\n✅ Tagged {len(master_makeup):,} makeup products')

Undertone tag coverage:
  undertone_warm         4.6%  (43 products)
  undertone_cool         1.3%  (12 products)
  undertone_neutral     94.2%  (877 products)

✅ Tagged 931 makeup products


---
## Step 6: Scoring Functions

In [10]:
# ── Skincare Scoring ──────────────────────────────────────────────────────────
#
# Score breakdown:
#   40% skin type match
#   40% concern match
#   20% quality (rating + popularity boost)

def score_skincare(df, skin_type, concerns):
    """
    Score each product for a user profile.
    
    Parameters
    ----------
    df         : master_skincare DataFrame
    skin_type  : str  — 'oily' | 'dry' | 'combination' | 'sensitive'
    concerns   : list — e.g. ['acne', 'dark_spots']
    
    Returns
    -------
    DataFrame sorted by score descending
    """
    scores = pd.Series(0.0, index=df.index)

    # 40% — Skin type match
    skin_col = f'skin_{skin_type}'
    if skin_col in df.columns:
        scores += df[skin_col].astype(float) * 0.40

    # 40% — Concern match (split equally across concerns)
    if concerns:
        concern_weight = 0.40 / len(concerns)
        for concern in concerns:
            concern_col = f'concern_{concern}'
            if concern_col in df.columns:
                scores += df[concern_col].astype(float) * concern_weight

    # 20% — Quality score (normalised rating)
    if 'rating' in df.columns:
        max_rating = df['rating'].max()
        if max_rating and max_rating > 0:
            norm_rating = df['rating'].fillna(0) / max_rating
            scores += norm_rating * 0.20

    result = df.copy()
    result['relevance_score'] = scores.round(4)
    result = result.sort_values('relevance_score', ascending=False)
    return result


# ── Makeup Scoring ────────────────────────────────────────────────────────────
#
# Score breakdown:
#   60% undertone match
#   20% product type match (user wants lipstick → boost lipsticks)
#   20% quality (rating)

def score_makeup(df, undertone, product_types=None):
    """
    Score each makeup product for a user's undertone.
    
    Parameters
    ----------
    df            : master_makeup DataFrame
    undertone     : str  — 'warm' | 'cool' | 'neutral'
    product_types : list — e.g. ['lipstick', 'blush'] or None for all
    
    Returns
    -------
    DataFrame sorted by score descending
    """
    scores = pd.Series(0.0, index=df.index)

    # 60% — Undertone match
    ut_col = f'undertone_{undertone}'
    if ut_col in df.columns:
        scores += df[ut_col].astype(float) * 0.60

    # 20% — Product type filter boost
    if product_types and 'product_type' in df.columns:
        type_match = df['product_type'].str.lower().apply(
            lambda t: any(pt.lower() in str(t) for pt in product_types)
        )
        scores += type_match.astype(float) * 0.20

    # 20% — Quality score
    if 'rating' in df.columns:
        max_rating = df['rating'].max()
        if max_rating and max_rating > 0:
            norm_rating = df['rating'].fillna(0) / max_rating
            scores += norm_rating * 0.20

    result = df.copy()
    result['relevance_score'] = scores.round(4)
    result = result.sort_values('relevance_score', ascending=False)
    return result


print('✅ Scoring functions defined')
print('   score_skincare(df, skin_type, concerns)')
print('   score_makeup(df, undertone, product_types)')

✅ Scoring functions defined
   score_skincare(df, skin_type, concerns)
   score_makeup(df, undertone, product_types)


---
## Step 7: Main Recommendation Engine Class

In [11]:
class PureGlowRecommendationEngine:
    """
    Core recommendation engine for PureGlow AI.

    Usage
    -----
    engine = PureGlowRecommendationEngine(master_skincare, master_makeup, COLOUR_CONFIG)

    recs = engine.recommend(
        skin_type  = 'oily',
        concerns   = ['acne', 'dark_spots'],
        undertone  = 'warm',
        skin_tone  = 'medium',
        top_n      = 5
    )
    """

    VALID_SKIN_TYPES = ['oily', 'dry', 'combination', 'sensitive']
    VALID_CONCERNS   = ['acne', 'dark_spots', 'redness', 'dullness', 'aging', 'dryness']
    VALID_UNDERTONES = ['warm', 'cool', 'neutral']

    def __init__(self, skincare_df, makeup_df, colour_config):
        self.skincare_df   = skincare_df
        self.makeup_df     = makeup_df
        self.colour_config = colour_config

    # ── Validation ────────────────────────────────────────────────────────────
    def _validate(self, skin_type, concerns, undertone):
        errors = []
        if skin_type and skin_type not in self.VALID_SKIN_TYPES:
            errors.append(f"Invalid skin_type '{skin_type}'. Choose from: {self.VALID_SKIN_TYPES}")
        if concerns:
            bad = [c for c in concerns if c not in self.VALID_CONCERNS]
            if bad:
                errors.append(f"Invalid concerns {bad}. Choose from: {self.VALID_CONCERNS}")
        if undertone and undertone not in self.VALID_UNDERTONES:
            errors.append(f"Invalid undertone '{undertone}'. Choose from: {self.VALID_UNDERTONES}")
        if errors:
            raise ValueError('\n'.join(errors))

    # ── Skincare recommendations ───────────────────────────────────────────────
    def get_skincare(self, skin_type, concerns=None, top_n=10, min_score=0.0):
        if self.skincare_df.empty:
            return pd.DataFrame()
        concerns = concerns or []
        self._validate(skin_type, concerns, None)
        scored = score_skincare(self.skincare_df, skin_type, concerns)
        result = scored[scored['relevance_score'] > min_score].head(top_n)
        return result[['product_id', 'name', 'brand', 'category', 'price',
                        'rating', 'relevance_score', 'url']].reset_index(drop=True)

    # ── Makeup recommendations ────────────────────────────────────────────────
    def get_makeup(self, undertone, product_types=None, top_n=10, min_score=0.0):
        if self.makeup_df.empty:
            return pd.DataFrame()
        self._validate(None, None, undertone)
        scored = score_makeup(self.makeup_df, undertone, product_types)
        keep_cols = [c for c in ['product_id', 'name', 'brand', 'product_type',
                                  'price', 'rating', 'hex_color', 'color_name',
                                  'relevance_score', 'url'] if c in scored.columns]
        return scored[scored['relevance_score'] > min_score].head(top_n)[keep_cols].reset_index(drop=True)

    # ── Colour palette ────────────────────────────────────────────────────────
    def get_colour_palette(self, undertone):
        self._validate(None, None, undertone)
        palette = self.colour_config.get(undertone, {})
        return {k: v for k, v in palette.items() if k != 'description'}

    # ── Full recommendation bundle ─────────────────────────────────────────────
    def recommend(self, skin_type, concerns=None, undertone=None,
                  skin_tone=None, top_n=5):
        self._validate(skin_type, concerns or [], undertone)
        result = {
            'user_profile': {
                'skin_type': skin_type,
                'concerns':  concerns or [],
                'undertone': undertone,
                'skin_tone': skin_tone,
            },
            'skincare_recommendations': [],
            'makeup_recommendations':   [],
            'colour_palette':           {},
        }

        # Skincare
        skincare_recs = self.get_skincare(skin_type, concerns, top_n=top_n)
        result['skincare_recommendations'] = skincare_recs.to_dict(orient='records')

        # Makeup + colour palette
        if undertone:
            makeup_recs = self.get_makeup(undertone, top_n=top_n)
            result['makeup_recommendations'] = makeup_recs.to_dict(orient='records')
            result['colour_palette'] = self.get_colour_palette(undertone)

        return result


# Instantiate
if not master_skincare.empty or not master_makeup.empty:
    engine = PureGlowRecommendationEngine(
        skincare_df   = master_skincare if not master_skincare.empty else pd.DataFrame(),
        makeup_df     = master_makeup   if not master_makeup.empty   else pd.DataFrame(),
        colour_config = COLOUR_CONFIG
    )
    print('✅ PureGlowRecommendationEngine instantiated!')
else:
    print('⚠️  Cannot instantiate engine — no data loaded')

✅ PureGlowRecommendationEngine instantiated!


---
## Step 8: Test the Engine

In [12]:
# ── Test Profile 1: Oily + Acne + Warm Undertone ─────────────────────────────
print('🧪 TEST 1: Oily skin, Acne + Dark Spots, Warm undertone\n')

if 'engine' in dir():
    recs = engine.recommend(
        skin_type = 'oily',
        concerns  = ['acne', 'dark_spots'],
        undertone = 'warm',
        skin_tone = 'medium',
        top_n     = 5
    )

    print(f'✓ Profile: {recs["user_profile"]}')

    print(f'\n🧴 Top {len(recs["skincare_recommendations"])} Skincare Recs:')
    if recs['skincare_recommendations']:
        display(pd.DataFrame(recs['skincare_recommendations'])
                  [['name', 'brand', 'category', 'price', 'rating', 'relevance_score']])
    else:
        print('  (No skincare results — check column mappings in Step 2)')

    print(f'\n💄 Top {len(recs["makeup_recommendations"])} Makeup Recs:')
    if recs['makeup_recommendations']:
        cols = [c for c in ['name', 'brand', 'product_type', 'color_name', 'price', 'relevance_score']
                if c in pd.DataFrame(recs['makeup_recommendations']).columns]
        display(pd.DataFrame(recs['makeup_recommendations'])[cols])
    else:
        print('  (No makeup results — check column mappings in Step 2)')

    print('\n🎨 Warm Colour Palette:')
    for product_type, shades in recs['colour_palette'].items():
        print(f'  {product_type:<15}: {", ".join(shades[:4])}')

🧪 TEST 1: Oily skin, Acne + Dark Spots, Warm undertone

✓ Profile: {'skin_type': 'oily', 'concerns': ['acne', 'dark_spots'], 'undertone': 'warm', 'skin_tone': 'medium'}

🧴 Top 5 Skincare Recs:


,name,brand,category,price,rating,relevance_score
0,Skin Perfecting Lotion - Blemish Prone/Oily Skin,Murad,skincare,NaN,NaN,0.6
1,pores no more¬Æ Mattifying Hydrator Pore Minimizing Gel,Dr. Brandt Skincare,skincare,NaN,NaN,0.6
2,Mattifying Primer With Anti-Acne Treatment,COVER FX,skincare,NaN,NaN,0.6
3,Bye Bye Pores Primer‚Ñ¢ Oil-Free Poreless Skin-Perfecting Serum Primer,IT Cosmetics,skincare,NaN,NaN,0.6
4,Ever-Matte Poreless Priming Perfector,BECCA,skincare,NaN,NaN,0.6



💄 Top 5 Makeup Recs:


,name,brand,product_type,color_name,price,relevance_score
0,Annabelle Biggy Bronzer Haute Gold,annabelle,bronzer,Annabelle Biggy Bronzer Haute Gold,11.99,0.8
1,Essie Encrusted Nail Polish Collection,essie,nail_polish,Essie Encrusted Nail Polish Collection,10.00,0.8
2,Maybelline Face Studio Master Hi-Light Light Booster Bronzer,maybelline,bronzer,Maybelline Face Studio Master Hi-Light Light Booster Bronzer,14.99,0.8
3,Earth Lab Loose Mineral Bronzer,NaN,bronzer,Earth Lab Loose Mineral Bronzer,24.00,0.8
4,Cargo Cosmetics Swimmables Water Resistant Bronzer,cargo cosmetics,bronzer,Cargo Cosmetics Swimmables Water Resistant Bronzer,29.00,0.8



🎨 Warm Colour Palette:
  blush          : peach, coral, terracotta, apricot
  lipstick       : coral, orange-red, brick, nude peach
  eyeshadow      : bronze, gold, copper, warm brown
  highlighter    : gold, champagne, bronze, warm pearl
  foundation     : yellow-based, warm beige, golden, warm ivory


In [13]:
# ── Test Profile 2: Dry + Redness + Cool Undertone ───────────────────────────
print('🧪 TEST 2: Dry skin, Redness + Dullness, Cool undertone\n')

if 'engine' in dir():
    recs2 = engine.recommend(
        skin_type = 'dry',
        concerns  = ['redness', 'dullness'],
        undertone = 'cool',
        top_n     = 5
    )

    print(f'🧴 Top Skincare:')
    if recs2['skincare_recommendations']:
        display(pd.DataFrame(recs2['skincare_recommendations'])
                  [['name', 'brand', 'category', 'price', 'rating', 'relevance_score']])

    print(f'\n🎨 Cool Colour Palette:')
    for product_type, shades in recs2['colour_palette'].items():
        print(f'  {product_type:<15}: {", ".join(shades[:4])}')

🧪 TEST 2: Dry skin, Redness + Dullness, Cool undertone

🧴 Top Skincare:


,name,brand,category,price,rating,relevance_score
0,Emerald Cannabis Sativa Hemp Seed Deep Moisture Glow Oil,Herbivore,skincare,NaN,NaN,0.6
1,The Ultimate Hydrating Vitamin C Facial Moisturizer,BeautyBio,skincare,NaN,NaN,0.6
2,HONEYMOON GLOW AHA Resurfacing Night Serum with Hydrating Honey + Gentle Flo...,Farmacy,skincare,NaN,NaN,0.6
3,Dry Erase¬Æ Ultra-Calming Face Cream,Jack Black,skincare,NaN,NaN,0.6
4,HONEYMOON GLOW AHA Resurfacing Night Serum with Hydrating Honey + Gentle Flo...,Farmacy,skincare,58.0,NaN,0.6



🎨 Cool Colour Palette:
  blush          : rose, pink, berry, mauve
  lipstick       : pink, berry, plum, fuchsia
  eyeshadow      : silver, taupe, cool brown, navy
  highlighter    : silver, icy pink, pearl, cool champagne
  foundation     : pink-based, cool beige, rosy, cool ivory


In [14]:
# ── Test Profile 3: Sensitive + Aging + Neutral Undertone ────────────────────
print('🧪 TEST 3: Sensitive skin, Aging + Dryness, Neutral undertone\n')

if 'engine' in dir():
    recs3 = engine.recommend(
        skin_type = 'sensitive',
        concerns  = ['aging', 'dryness'],
        undertone = 'neutral',
        top_n     = 5
    )

    print(f'🧴 Top Skincare:')
    if recs3['skincare_recommendations']:
        display(pd.DataFrame(recs3['skincare_recommendations'])
                  [['name', 'brand', 'category', 'price', 'rating', 'relevance_score']])

🧪 TEST 3: Sensitive skin, Aging + Dryness, Neutral undertone

🧴 Top Skincare:


,name,brand,category,price,rating,relevance_score
0,Hypoallergenic Firming Eye Cream,Perricone MD,skincare,NaN,NaN,0.6
1,HONEYMOON GLOW AHA Resurfacing Night Serum with Hydrating Honey + Gentle Flo...,Farmacy,skincare,58.0,NaN,0.6
2,Skin Soothing Hydrating Lotion,Dermalogica,skincare,22.0,NaN,0.6
3,HONEYMOON GLOW AHA Resurfacing Night Serum with Hydrating Honey + Gentle Flo...,Farmacy,skincare,NaN,NaN,0.6
4,Dry Erase¬Æ Ultra-Calming Face Cream,Jack Black,skincare,NaN,NaN,0.6


---
## Step 9: Save Model Outputs

In [15]:
print('Saving model outputs...\n')

# Save master product tables
if not master_skincare.empty:
    path = MODELS_DIR / 'skincare_products.csv'
    master_skincare.to_csv(path, index=False)
    print(f'  💾 skincare_products.csv     → {len(master_skincare):,} products')

if not master_makeup.empty:
    path = MODELS_DIR / 'makeup_products.csv'
    master_makeup.to_csv(path, index=False)
    print(f'  💾 makeup_products.csv       → {len(master_makeup):,} products')

# Save keyword maps as JSON (for API)
maps_path = MODELS_DIR / 'keyword_maps.json'
with open(maps_path, 'w') as f:
    json.dump({
        'skin_type_keywords': SKIN_TYPE_KEYWORDS,
        'concern_keywords':   CONCERN_KEYWORDS,
        'undertone_keywords': UNDERTONE_KEYWORDS,
    }, f, indent=2)
print(f'  💾 keyword_maps.json')

# Save colour config
print(f'  💾 colour_config.json')

print(f'\n  📁 All saved to: {MODELS_DIR}')

Saving model outputs...

  💾 skincare_products.csv     → 3,474 products
  💾 makeup_products.csv       → 931 products
  💾 keyword_maps.json
  💾 colour_config.json

  📁 All saved to: C:\Users\HP\OneDrive\Desktop\PureGlow AI\models


In [16]:
# ── Export recommendation_engine.py for API use ───────────────────────────────
engine_code = '''
"""
PureGlow AI — Recommendation Engine
Auto-generated from 03_recommendation_model.ipynb

Usage in FastAPI:
    from recommendation_engine import PureGlowRecommendationEngine, load_engine
    engine = load_engine()
    recs = engine.recommend(skin_type='oily', concerns=['acne'], undertone='warm')
"""
import json
import numpy as np
import pandas as pd
from pathlib import Path

MODELS_DIR = Path(__file__).parent.parent / 'models'

SKIN_TYPE_KEYWORDS = {
    'oily':        ['oily', 'oil control', 'oil-free', 'mattify', 'matte'],
    'dry':         ['dry', 'hydrating', 'hydration', 'moisture', 'moisturising', 'nourishing'],
    'combination': ['combination', 'balanced', 'normal'],
    'sensitive':   ['sensitive', 'gentle', 'soothing', 'calming', 'fragrance-free'],
}

CONCERN_KEYWORDS = {
    'acne':       ['acne', 'breakout', 'blemish', 'pore', 'salicylic', 'benzoyl'],
    'dark_spots': ['dark spot', 'brightening', 'vitamin c', 'niacinamide', 'hyperpigmentation'],
    'redness':    ['redness', 'rosacea', 'centella', 'calming', 'soothing'],
    'dullness':   ['dull', 'glow', 'radiant', 'brightening', 'exfoliat', 'aha', 'bha'],
    'aging':      ['anti-aging', 'retinol', 'firming', 'collagen', 'peptide'],
    'dryness':    ['dry', 'moisture', 'hydrat', 'hyaluronic', 'ceramide'],
}

UNDERTONE_KEYWORDS = {
    'warm':    ['warm', 'golden', 'peach', 'coral', 'bronze', 'copper', 'terracotta', 'gold'],
    'cool':    ['cool', 'pink', 'berry', 'plum', 'silver', 'icy', 'rose', 'lavender'],
    'neutral': ['nude', 'natural', 'neutral', 'taupe', 'beige', 'soft', 'universal'],
}


def _tag_column(series, keyword_map):
    text = series.fillna(\'\').str.lower()
    return {tag: text.apply(lambda t: any(kw in t for kw in kws))
            for tag, kws in keyword_map.items()}


def score_skincare(df, skin_type, concerns):
    scores = pd.Series(0.0, index=df.index)
    skin_col = f\'skin_{skin_type}\'
    if skin_col in df.columns:
        scores += df[skin_col].astype(float) * 0.40
    if concerns:
        w = 0.40 / len(concerns)
        for c in concerns:
            col = f\'concern_{c}\'
            if col in df.columns:
                scores += df[col].astype(float) * w
    if \'rating\' in df.columns:
        mx = df[\'rating\'].max()
        if mx:
            scores += (df[\'rating\'].fillna(0) / mx) * 0.20
    df = df.copy()
    df[\'relevance_score\'] = scores.round(4)
    return df.sort_values(\'relevance_score\', ascending=False)


def score_makeup(df, undertone, product_types=None):
    scores = pd.Series(0.0, index=df.index)
    ut_col = f\'undertone_{undertone}\'
    if ut_col in df.columns:
        scores += df[ut_col].astype(float) * 0.60
    if product_types and \'product_type\' in df.columns:
        match = df[\'product_type\'].str.lower().apply(
            lambda t: any(pt.lower() in str(t) for pt in product_types))
        scores += match.astype(float) * 0.20
    if \'rating\' in df.columns:
        mx = df[\'rating\'].max()
        if mx:
            scores += (df[\'rating\'].fillna(0) / mx) * 0.20
    df = df.copy()
    df[\'relevance_score\'] = scores.round(4)
    return df.sort_values(\'relevance_score\', ascending=False)


class PureGlowRecommendationEngine:
    VALID_SKIN_TYPES = [\'oily\', \'dry\', \'combination\', \'sensitive\']
    VALID_CONCERNS   = [\'acne\', \'dark_spots\', \'redness\', \'dullness\', \'aging\', \'dryness\']
    VALID_UNDERTONES = [\'warm\', \'cool\', \'neutral\']

    def __init__(self, skincare_df, makeup_df, colour_config):
        self.skincare_df   = skincare_df
        self.makeup_df     = makeup_df
        self.colour_config = colour_config

    def get_skincare(self, skin_type, concerns=None, top_n=10):
        concerns = concerns or []
        scored = score_skincare(self.skincare_df, skin_type, concerns)
        cols = [c for c in [\'product_id\',\'name\',\'brand\',\'category\',\'price\',\'rating\',\'relevance_score\',\'url\'] if c in scored.columns]
        return scored.head(top_n)[cols].reset_index(drop=True)

    def get_makeup(self, undertone, product_types=None, top_n=10):
        scored = score_makeup(self.makeup_df, undertone, product_types)
        cols = [c for c in [\'product_id\',\'name\',\'brand\',\'product_type\',\'price\',\'rating\',\'hex_color\',\'color_name\',\'relevance_score\',\'url\'] if c in scored.columns]
        return scored.head(top_n)[cols].reset_index(drop=True)

    def get_colour_palette(self, undertone):
        p = self.colour_config.get(undertone, {})
        return {k: v for k, v in p.items() if k != \'description\'}

    def recommend(self, skin_type, concerns=None, undertone=None, skin_tone=None, top_n=5):
        return {
            \'user_profile\': {\'skin_type\': skin_type, \'concerns\': concerns or [],
                              \'undertone\': undertone, \'skin_tone\': skin_tone},
            \'skincare_recommendations\': self.get_skincare(skin_type, concerns, top_n).to_dict(\'records\'),
            \'makeup_recommendations\'  : self.get_makeup(undertone, top_n=top_n).to_dict(\'records\') if undertone else [],
            \'colour_palette\'          : self.get_colour_palette(undertone) if undertone else {},
        }


def load_engine() -> PureGlowRecommendationEngine:
    """Load saved model files and return a ready-to-use engine."""
    skincare_path = MODELS_DIR / \'skincare_products.csv\'
    makeup_path   = MODELS_DIR / \'makeup_products.csv\'
    colour_path   = MODELS_DIR / \'colour_config.json\'

    skincare_df = pd.read_csv(skincare_path) if skincare_path.exists() else pd.DataFrame()
    makeup_df   = pd.read_csv(makeup_path)   if makeup_path.exists()   else pd.DataFrame()
    with open(colour_path) as f:
        colour_config = json.load(f)

    return PureGlowRecommendationEngine(skincare_df, makeup_df, colour_config)
'''

engine_path = SRC_DIR / 'recommendation_engine.py'
with open(engine_path, 'w') as f:
    f.write(engine_code.strip())

print(f'✅ Saved recommendation_engine.py → {engine_path}')
print('\n📌 Import in FastAPI with:')
print('   from recommendation_engine import load_engine')
print('   engine = load_engine()')

✅ Saved recommendation_engine.py → C:\Users\HP\OneDrive\Desktop\PureGlow AI\src\recommendation_engine.py

📌 Import in FastAPI with:
   from recommendation_engine import load_engine
   engine = load_engine()


In [17]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print('\n' + '═'*70)
print('  ✅ RECOMMENDATION MODEL COMPLETE')
print('═'*70)
print()

models_files = list(MODELS_DIR.glob('*'))
src_files    = list(SRC_DIR.glob('*.py'))

print('📁 models/')
for f in sorted(models_files):
    size = f.stat().st_size / 1024
    print(f'   {f.name:<40} {size:>8.1f} KB')

print('\n📁 src/')
for f in sorted(src_files):
    print(f'   {f.name}')

print()
print('📌 Next step → 04_fastapi_backend.py (plug engine into the API)')
print()
print('Quick test the engine loads correctly:')
print('   from src.recommendation_engine import load_engine')
print('   engine = load_engine()')
print('   recs = engine.recommend(skin_type="oily", concerns=["acne"], undertone="warm")')


══════════════════════════════════════════════════════════════════════
  ✅ RECOMMENDATION MODEL COMPLETE
══════════════════════════════════════════════════════════════════════

📁 models/
   colour_config.json                            2.1 KB
   keyword_maps.json                             2.2 KB
   makeup_products.csv                         196.1 KB
   skincare_products.csv                       525.7 KB

📁 src/
   recommendation_engine.py

📌 Next step → 04_fastapi_backend.py (plug engine into the API)

Quick test the engine loads correctly:
   from src.recommendation_engine import load_engine
   engine = load_engine()
   recs = engine.recommend(skin_type="oily", concerns=["acne"], undertone="warm")
